In [67]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

In [68]:

def auc_confidence_interval(y_true, y_pred, num_bootstrap=1000, alpha=0.05, random_state=None):
    """
    Compute the 95% confidence interval for the AUC using bootstrapping.

    Parameters:
    y_true (array-like): True binary labels.
    y_pred (array-like): Predicted scores.
    num_bootstrap (int): Number of bootstrap samples (default: 1000).
    alpha (float): Significance level (default: 0.05).
    random_state (int, optional): Random seed for reproducibility.

    Returns:
    tuple: (AUC, lower bound, upper bound)
    """
    if random_state is not None:
        np.random.seed(random_state)

    # Calculate the base AUC
    auc_value = roc_auc_score(y_true, y_pred)

    # Bootstrap resampling
    auc_bootstrap = []
    for _ in range(num_bootstrap):
        indices = np.random.choice(range(len(y_true)), len(y_true), replace=True)
        if len(np.unique(y_true[indices])) < 2:
            continue  # Skip if only one class is present in resample

        auc_bootstrap.append(roc_auc_score(y_true[indices], y_pred[indices]))

    # Calculate the confidence interval
    lower = np.percentile(auc_bootstrap, 100 * alpha / 2)
    upper = np.percentile(auc_bootstrap, 100 * (1 - alpha / 2))

    return auc_value, lower, upper


y_true = np.array([0, 0, 1, 1])
y_pred = np.array([0.1, 0.4, 0.35, 0.8])

auc, lower, upper = auc_confidence_interval(y_true, y_pred)
print(f"AUC: {auc:.4f}, 95% CI: [{lower:.4f}, {upper:.4f}]")

AUC: 0.7500, 95% CI: [0.0000, 1.0000]


In [ ]:

# 

preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\LR_ensemble_preds.csv"
preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\LR_ensemble_preds_MDACC.csv"


# preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial32_singletox_models\all_ST_models_ens_predictions.csv"


# preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32\ensemble_predictions.csv"

preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_May25\Trial_32\ensemble_predictions_MDACC.csv"

preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_May25\single_tox_models\all_ST_models_ens_predictions_MDACC.csv"


df_preds = pd.read_csv(preds_csv_dir, sep=';')
df_preds['PatientID'] = df_preds['PatientID'].astype(str).str.zfill(7)

df_predictions_only = df_preds[['Aspiration_M06_pred', 'Dysphagia_M06_pred', 'Sticky_M06_pred', 'Taste_M06_pred', 'Xerostomia_M06_pred']].copy()
df_predictions_only.columns = df_predictions_only.columns.str.replace('_pred', '', regex=False)

df_labels_only = df_preds[['Aspiration_M06_true', 'Dysphagia_M06_true', 'Sticky_M06_true', 'Taste_M06_true', 'Xerostomia_M06_true']].copy()
df_labels_only.columns = df_labels_only.columns.str.replace('_true', '', regex=False)



In [70]:
for col in df_predictions_only.columns:
    y_true = df_labels_only[col].values
    y_pred = df_predictions_only[col].values

    valid_indices = y_true != -1
    y_true = y_true[valid_indices]
    y_pred = y_pred[valid_indices]

    auc, lower, upper = auc_confidence_interval(y_true, y_pred)
    print(f"{col} AUC: {auc:.4f}, 95% CI: [{lower:.4f}, {upper:.4f}]")

Aspiration_M06 AUC: 0.6714, 95% CI: [0.5355, 0.8095]
Dysphagia_M06 AUC: 0.6436, 95% CI: [0.5674, 0.7134]
Sticky_M06 AUC: 0.6079, 95% CI: [0.5295, 0.6949]
Taste_M06 AUC: 0.5839, 95% CI: [0.5211, 0.6461]
Xerostomia_M06 AUC: 0.5915, 95% CI: [0.5305, 0.6556]


In [71]:
# compute the brier score for each endpoint

from sklearn.metrics import brier_score_loss
from sklearn.metrics import  r2_score

for col in df_predictions_only.columns:
    y_true = df_labels_only[col].values
    y_pred = df_predictions_only[col].values

    valid_indices = y_true != -1
    y_true = y_true[valid_indices]
    y_pred = y_pred[valid_indices]

    brier_score = brier_score_loss(y_true, y_pred)


    r2 = r2_score(y_true, y_pred)
    print(f"{col} Brier score: {brier_score:.4f}  R2 score: {r2:.4f}")

Aspiration_M06 Brier score: 0.0522  R2 score: 0.0136
Dysphagia_M06 Brier score: 0.1750  R2 score: 0.0669
Sticky_M06 Brier score: 0.1829  R2 score: -0.0618
Taste_M06 Brier score: 0.2371  R2 score: 0.0104
Xerostomia_M06 Brier score: 0.2482  R2 score: -0.0129


In [ ]:
# merge all of the single toxictiy model prediction csvs into one file
root_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial32_singletox_models"
root_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_May25\single_tox_models"


toxicity_models = ['Aspiration_M06', 'Dysphagia_M06', 'Sticky_M06', 'Taste_M06', 'Xerostomia_M06']
df_all_preds = None
for model in toxicity_models:
    print(model)

    preds_csv_dir = f"{root_dir}\\{model}\\ensemble_predictions_MDACC.csv"
    df_preds = pd.read_csv(preds_csv_dir, sep=';')
    df_preds['PatientID'] = df_preds['PatientID'].astype(str).str.zfill(7)
    #df_preds = df_preds.rename(columns={f"{model}_pred": model}

    if df_all_preds is None:
        df_all_preds = df_preds.copy()
    else:
        df_all_preds = pd.merge(df_all_preds, df_preds, on=['PatientID', 'Mode'], how='left')

df_all_preds = df_all_preds.loc[:, ~df_all_preds.columns.duplicated()]
df_all_preds.to_csv(f"{root_dir}\\all_ST_models_ens_predictions_MDACC.csv", sep=';', index=False)

Sticky_M06


FileNotFoundError: [Errno 2] No such file or directory: '\\\\zkh\\appdata\\RTDicom\\Projectline_HNC_modelling\\Users\\Daniel MacRae\\1. MultiTox_HNC\\Feb_2025_results\\Trial32_endpoint_combinations\\salivary_domain\\Sticky_M06\\ensemble_predictions_MDACC.csv'

# Development Set

In [101]:
import os 


preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32"

# preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_May25\single_tox_models
# 
# ST_folders = [x for x in os.listdir(preds_csv_dir) if os.path.isdir(os.path.join(preds_csv_dir, x))]
# for ST_folder in ST_folders:
#     ST_folder_dir = os.path.join(preds_csv_dir, ST_folder)
#     folders = [x for x in os.listdir(ST_folder_dir) if os.path.isdir(os.path.join(ST_folder_dir, x))]

folders = [x for x in os.listdir(preds_csv_dir) if os.path.isdir(os.path.join(preds_csv_dir, x))]

CI_dict = {}

for folder in folders:
    # validation preds
    df_dir = os.path.join(preds_csv_dir, folder, "all_lr_predictions.csv")
    df_all_preds = pd.read_csv(df_dir, sep=';')
    # get only the validation set
    df_validation = df_all_preds[df_all_preds['Mode'] == 'val']

    print(len(df_validation))

    CI_dict[folder] = {}

    # compute the confidernce intervals for the AUC
    for col in list(df_validation.columns):
        temp_dict = {}
        if col.endswith('_pred'):
            y_true = df_validation[col.replace('_pred', '_true')].values
            y_pred = df_validation[col].values

            valid_indices = y_true != -1
            y_true = y_true[valid_indices]
            y_pred = y_pred[valid_indices]

            auc, lower, upper = auc_confidence_interval(y_true, y_pred)
            temp_dict[f"{col}"] = {'AUC' : auc, 'lower' : lower, 'upper' : upper}
            print(f"{col} AUC: {auc:.4f}, 95% CI: [{lower:.4f}, {upper:.4f}]")

            CI_dict[folder][col] = temp_dict


mean_dict = {}

for fold, endpoints in CI_dict.items():
    for endpoint, metrics in endpoints.items():
        for metric, values in metrics.items():
            if metric not in mean_dict:
                mean_dict[metric] = {'AUC': [], 'lower': [], 'upper': []}
            mean_dict[metric]['AUC'].append(values['AUC'])
            mean_dict[metric]['lower'].append(values['lower'])
            mean_dict[metric]['upper'].append(values['upper'])

# Compute the mean for each metric
for metric, values in mean_dict.items():
    mean_dict[metric] = {key: round(np.mean(val), 2) for key, val in values.items()}

print(mean_dict)

168
Aspiration_M06_pred AUC: 0.8312, 95% CI: [0.7098, 0.9276]
Dysphagia_M06_pred AUC: 0.7925, 95% CI: [0.7146, 0.8648]
Sticky_M06_pred AUC: 0.7105, 95% CI: [0.6298, 0.7915]
Taste_M06_pred AUC: 0.7242, 95% CI: [0.6250, 0.8125]
Xerostomia_M06_pred AUC: 0.7598, 95% CI: [0.6775, 0.8348]
168
Aspiration_M06_pred AUC: 0.6417, 95% CI: [0.4969, 0.7827]
Dysphagia_M06_pred AUC: 0.8360, 95% CI: [0.7583, 0.9090]
Sticky_M06_pred AUC: 0.6757, 95% CI: [0.5677, 0.7683]
Taste_M06_pred AUC: 0.7450, 95% CI: [0.6642, 0.8171]
Xerostomia_M06_pred AUC: 0.7505, 95% CI: [0.6701, 0.8280]
168
Aspiration_M06_pred AUC: 0.8312, 95% CI: [0.7002, 0.9449]
Dysphagia_M06_pred AUC: 0.7871, 95% CI: [0.7079, 0.8551]
Sticky_M06_pred AUC: 0.7271, 95% CI: [0.6430, 0.8068]
Taste_M06_pred AUC: 0.7497, 95% CI: [0.6631, 0.8261]
Xerostomia_M06_pred AUC: 0.7307, 95% CI: [0.6458, 0.8104]
168
Aspiration_M06_pred AUC: 0.6601, 95% CI: [0.4315, 0.8597]
Dysphagia_M06_pred AUC: 0.7953, 95% CI: [0.7126, 0.8691]
Sticky_M06_pred AUC: 0.6422, 

In [100]:
mean_dict

{'Aspiration_M06_pred': {'AUC': 0.74, 'lower': 0.58, 'upper': 0.88},
 'Dysphagia_M06_pred': {'AUC': 0.8, 'lower': 0.73, 'upper': 0.87},
 'Sticky_M06_pred': {'AUC': 0.7, 'lower': 0.61, 'upper': 0.78},
 'Taste_M06_pred': {'AUC': 0.71, 'lower': 0.61, 'upper': 0.8},
 'Xerostomia_M06_pred': {'AUC': 0.75, 'lower': 0.67, 'upper': 0.82}}

In [96]:
preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_May25\single_tox_models"

folders = [x for x in os.listdir(preds_csv_dir) if os.path.isdir(os.path.join(preds_csv_dir, x))]

folders

['Aspiration_M06',
 'Dysphagia_M06',
 'Sticky_M06',
 'Taste_M06',
 'Xerostomia_M06']

In [113]:
import os 

preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_results\Trial_32"

preds_csv_dir = r"\\zkh\appdata\RTDicom\Projectline_HNC_modelling\Users\Daniel MacRae\1. MultiTox_HNC\Feb_2025_May25\single_tox_models"

ST_folders = [x for x in os.listdir(preds_csv_dir) if os.path.isdir(os.path.join(preds_csv_dir, x))]
for ST_folder in ST_folders:
    ST_folder_dir = os.path.join(preds_csv_dir, ST_folder)
    folders = [x for x in os.listdir(ST_folder_dir) if os.path.isdir(os.path.join(ST_folder_dir, x))]


#folders = [x for x in os.listdir(preds_csv_dir) if os.path.isdir(os.path.join(preds_csv_dir, x))]

    brier_scores_dict = {}

    for folder in folders:
        # validation preds
        df_dir = os.path.join(ST_folder_dir, folder, "model_predictions.csv")
        df_all_preds = pd.read_csv(df_dir, sep=';')
        # get only the validation set
        df_validation = df_all_preds[df_all_preds['Mode'] == 'val']

        # print(len(df_validation))

        brier_scores_dict[folder] = {}

        # compute the Brier score for each endpoint
        for col in list(df_validation.columns):
            if col.endswith('_pred'):
                y_true = df_validation[col.replace('_pred', '_true')].values
                y_pred = df_validation[col].values

                valid_indices = y_true != -1
                y_true = y_true[valid_indices]
                y_pred = y_pred[valid_indices]

                brier_score = brier_score_loss(y_true, y_pred)
                # brier_score = r2_score(y_true, y_pred)
                brier_scores_dict[folder][col] = brier_score
                print(f"{col} Brier score: {brier_score:.4f}")

    mean_brier_scores = {}

    for fold, endpoints in brier_scores_dict.items():
        for endpoint, brier_score in endpoints.items():
            if endpoint not in mean_brier_scores:
                mean_brier_scores[endpoint] = []
            mean_brier_scores[endpoint].append(brier_score)

    # Compute the mean Brier score for each endpoint
    for endpoint, scores in mean_brier_scores.items():
        mean_brier_scores[endpoint] = round(np.mean(scores), 4)

    print(mean_brier_scores)


Aspiration_M06_pred Brier score: 0.0652
Aspiration_M06_pred Brier score: 0.0651
Aspiration_M06_pred Brier score: 0.0604
Aspiration_M06_pred Brier score: 0.0669
Aspiration_M06_pred Brier score: 0.0602
{'Aspiration_M06_pred': 0.0636}
Dysphagia_M06_pred Brier score: 0.1513
Dysphagia_M06_pred Brier score: 0.1319
Dysphagia_M06_pred Brier score: 0.1370
Dysphagia_M06_pred Brier score: 0.1438
Dysphagia_M06_pred Brier score: 0.1437
{'Dysphagia_M06_pred': 0.1415}
Sticky_M06_pred Brier score: 0.2003
Sticky_M06_pred Brier score: 0.1840
Sticky_M06_pred Brier score: 0.2134
Sticky_M06_pred Brier score: 0.1895
Sticky_M06_pred Brier score: 0.2027
{'Sticky_M06_pred': 0.198}
Taste_M06_pred Brier score: 0.1662
Taste_M06_pred Brier score: 0.1768
Taste_M06_pred Brier score: 0.1716
Taste_M06_pred Brier score: 0.1629
Taste_M06_pred Brier score: 0.1522
{'Taste_M06_pred': 0.1659}
Xerostomia_M06_pred Brier score: 0.2108
Xerostomia_M06_pred Brier score: 0.1986
Xerostomia_M06_pred Brier score: 0.1785
Xerostomia_M0